# Diagnostic 1A — production accuracy figures

Outputs: `research/diagnostics/figures/` (sibling of this folder).
Shared geometry: `BAR_W=0.3`, `PAIR_GAP=0.025`, `INNER_GAP=0.55`.
Legend frame restored. No figure titles (use LaTeX captions).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

out_dir = Path("..") / "figures"
out_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.facecolor": "white",
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
})

BAR_W = 0.3
PAIR_GAP = 0.025
PAIR_STEP = BAR_W + PAIR_GAP
INNER_GAP = 0.55
FAMILY_GAP = 0.9
FIG_W = 8.8
FIG_H_SLOTS = 3.4
FIG_H_REG = 3.65 * 0.75  # 2.7375; 3/4 of published height


def style_ax(ax, ylabel=False, box=False):
    ax.set_ylim(0, 108)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.yaxis.grid(True, linestyle="--", linewidth=0.6, alpha=0.45, zorder=0)
    ax.set_axisbelow(True)
    if box:
        for side in ("top", "right", "bottom", "left"):
            ax.spines[side].set_visible(True)
    else:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    if ylabel:
        ax.set_ylabel("Production accuracy (%)")


def label_bars(bars, fontsize=8):
    for bar in bars:
        h = bar.get_height()
        bar.axes.text(
            bar.get_x() + bar.get_width() / 2,
            h + 0.9,
            f"{h:.0f}%",
            ha="center",
            va="bottom",
            fontsize=fontsize,
            zorder=5,
        )


off_l, off_r = -PAIR_STEP / 2, PAIR_STEP / 2

models = {
    "1.7B": {"past": {"EN": 74, "ES": 52}, "participle": {"EN": 81, "ES": 85}},
    "4B": {"past": {"EN": 89, "ES": 69}, "participle": {"EN": 86, "ES": 89}},
}
tenses = ["past", "participle"]
lang_colours = {"EN": "C0", "ES": "C1"}
lang_labels = {"EN": "English", "ES": "Spanish"}
tense_centre_gap = PAIR_STEP + INNER_GAP
x = np.array([0.0, tense_centre_gap])

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H_SLOTS), sharey=True)
for ax, (model_name, tense_data) in zip(axes, models.items()):
    for lang, off in zip(["EN", "ES"], [off_l, off_r]):
        vals = [tense_data[t][lang] for t in tenses]
        bars = ax.bar(
            x + off, vals, width=BAR_W,
            label=lang_labels[lang], color=lang_colours[lang], zorder=3,
        )
        label_bars(bars)
    ax.set_title(f"Qwen3 {model_name}", fontsize=10, pad=4)
    ax.set_xticks(x)
    ax.set_xticklabels(["Past / preterite", "Participle"])
    ax.set_xlim(x[0] + off_l - BAR_W / 2 - 0.12, x[-1] + off_r + BAR_W / 2 + 0.12)
    style_ax(ax, ylabel=(ax is axes[0]), box=True)

axes[0].legend(loc="upper left", frameon=True, fancybox=False, edgecolor="0.6")
fig.subplots_adjust(left=0.09, right=0.99, top=0.88, bottom=0.14, wspace=0.22)
fig.savefig(out_dir / "diag1a_accuracy_by_slot.png")
plt.show()

regular_colour = "#2ca02c"
irregular_colour = "#d62728"
past_items = [("EN past", 83, 65), ("ES preterite", 76, 28)]
part_items = [("EN participle", 88, 75), ("ES participle", 89, 81)]

positions, labels, reg_vals, irreg_vals = [], [], [], []
cursor = 0.0
for i, (lab, r, irr) in enumerate(past_items):
    if i:
        cursor += INNER_GAP
    pair_centre = cursor + PAIR_STEP / 2
    positions.append(pair_centre)
    labels.append(lab)
    reg_vals.append(r)
    irreg_vals.append(irr)
    cursor += PAIR_STEP

past_centre = float(np.mean(positions))
cursor += FAMILY_GAP
part_positions = []
for i, (lab, r, irr) in enumerate(part_items):
    if i:
        cursor += INNER_GAP
    pair_centre = cursor + PAIR_STEP / 2
    positions.append(pair_centre)
    part_positions.append(pair_centre)
    labels.append(lab)
    reg_vals.append(r)
    irreg_vals.append(irr)
    cursor += PAIR_STEP

part_centre = float(np.mean(part_positions))
positions = np.array(positions)

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H_REG))
reg_bars = ax.bar(positions + off_l, reg_vals, width=BAR_W, color=regular_colour, label="Regular", zorder=3)
irreg_bars = ax.bar(positions + off_r, irreg_vals, width=BAR_W, color=irregular_colour, label="Irregular", zorder=3)
label_bars(reg_bars, fontsize=10)
label_bars(irreg_bars, fontsize=10)

ax.set_xticks(positions)
ax.set_xticklabels(labels)
style_ax(ax, ylabel=True)
ax.set_ylim(0, 122)
ax.tick_params(axis="both", labelsize=11)
ax.yaxis.label.set_size(12)
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.06), frameon=True, fancybox=False, edgecolor="0.6", fontsize=11)
ax.annotate("Past / preterite", xy=(past_centre, 0), xytext=(0, -24),
            textcoords="offset points", ha="center", va="top", fontsize=11)
ax.annotate("Participle", xy=(part_centre, 0), xytext=(0, -24),
            textcoords="offset points", ha="center", va="top", fontsize=11)
ax.axvline((positions[1] + positions[2]) / 2, color="0.8", linestyle=":", linewidth=0.9, zorder=0)
ax.set_xlim(positions[0] + off_l - BAR_W / 2 - 0.15, positions[-1] + off_r + BAR_W / 2 + 0.15)
fig.subplots_adjust(left=0.07, right=0.995, top=0.98, bottom=0.16)
fig.savefig(out_dir / "diag1a_accuracy_by_regularity.png", bbox_inches="tight", pad_inches=0.04)
plt.show()


